In [ ]:
# ── CELL 1: Imports + Config ──────────────────────────────────────────────────

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path

import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

# ── Config — edit these ───────────────────────────────────────────────────────
OUTER_ZIP     = "/Users/sabare/Downloads/texas_pdq.zip"          # your outer zip
INNER_ZIP     = "PDQ_DSV.zip"                                     # zip inside outer zip
CYCLE_PARQUET = "./pdq_lease_output/og_lease_cycle.parquet"       # your saved production file
SHAPEFILE_DIR = "./well_layers"                                    # folder with RRC shapefiles
OUT_DIR       = "./pdq_lease_output"                              # where to save output
FORMAT        = "parquet"                                         # "parquet" or "csv"
OIL_GAS_FILTER = "O"                                              # must match what you used before
CHUNKSIZE     = 75_000

# Constants
DELIMITER  = "}"
ENCODING   = "latin-1"
WELL_FILE  = "OG_WELL_COMPLETION_DATA_TABLE.dsv"

WELL_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",
    "WELL_NO",
    "API_COUNTY_CODE",
    "API_UNIQUE_NO",
    "COUNTY_NAME",
    "WELLBORE_LOCATION_CODE",
    "WELL_SHUTIN_DT",
]

log.info("✓ Cell 1 done — imports and config loaded.")


In [ ]:
# ── CELL 2: Well completion cleaning + reader functions ───────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def clean_well_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in WELL_KEEP if c in chunk.columns]]
    # Build full API number
    if "API_COUNTY_CODE" in chunk.columns and "API_UNIQUE_NO" in chunk.columns:
        chunk["API_NO"] = (
            chunk["API_COUNTY_CODE"].str.strip().str.zfill(3) +
            chunk["API_UNIQUE_NO"].str.strip().str.zfill(5)
        )
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "WELLBORE_LOCATION_CODE"):
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("category")
    return chunk

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found.\nAvailable: {outer_contents}")
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Cell 2 done — functions defined.")


In [ ]:
# ── CELL 3: Load OG_WELL_COMPLETION directly from zip ────────────────────────

log.info("\n── Loading OG_WELL_COMPLETION from zip ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_well = read_chunked(inner_zf, WELL_FILE, clean_well_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_well.shape)
print("Memory :", f"{df_well.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_well.columns.tolist())
print("\nSample:")
print(df_well[["DISTRICT_NO", "LEASE_NO", "WELL_NO", "API_NO", "COUNTY_NAME"]].head(10))


In [ ]:
# ── CELL 4: Load saved production parquet ────────────────────────────────────

log.info("Loading og_lease_cycle.parquet ...")
df_cycle = pd.read_parquet(CYCLE_PARQUET)
log.info("  Shape : %s", df_cycle.shape)
log.info("  Memory: %.1f MB", df_cycle.memory_usage(deep=True).sum() / 1_048_576)
print("Columns:", df_cycle.columns.tolist())
df_cycle.head(3)


In [ ]:
# ── CELL 5: Load RRC Well Layers shapefiles ───────────────────────────────────
# Download from:
# https://mft.rrc.texas.gov/link/d551fb20-442e-4b67-84fa-ac3f23ecabb4
# Extract to the folder set in SHAPEFILE_DIR in Cell 1.

shp_files = list(Path(SHAPEFILE_DIR).rglob("*.shp"))
log.info("Found %d shapefiles in %s", len(shp_files), SHAPEFILE_DIR)

if len(shp_files) == 0:
    raise FileNotFoundError(
        f"No .shp files found in {SHAPEFILE_DIR}.\n"
        f"Download and extract the Well Layers shapefile from:\n"
        f"https://mft.rrc.texas.gov/link/d551fb20-442e-4b67-84fa-ac3f23ecabb4"
    )

gdfs = []
for shp in shp_files:
    gdf = gpd.read_file(shp)
    gdfs.append(gdf)
    log.info("  Loaded %s (%s rows)", shp.name, len(gdf))

wells_geo = pd.concat(gdfs, ignore_index=True)
del gdfs
gc.collect()

log.info("✓ Combined shapefile: %s rows x %s cols", len(wells_geo), len(wells_geo.columns))
print("\nShapefile columns:", wells_geo.columns.tolist())
print("\nSample:")
print(wells_geo.head(3))


In [ ]:
# ── CELL 6: Extract coordinates from shapefile ────────────────────────────────
# Check column names printed above and update API_COL if needed.
# Common names: "API", "API_NUMBER", "API_NO", "API_NUM"

API_COL = "API"   # ← update if shapefile uses a different column name

coords = wells_geo[[API_COL, "geometry"]].copy()
coords["LATITUDE"]  = coords.geometry.y
coords["LONGITUDE"] = coords.geometry.x
coords = coords.drop(columns=["geometry"])
coords = coords.rename(columns={API_COL: "API_NO"})
coords["API_NO"] = coords["API_NO"].astype(str).str.strip().str.zfill(8)

del wells_geo
gc.collect()

log.info("✓ Coords: %s rows", len(coords))
print("Sample:")
print(coords.head(10))
print("\nNulls:", coords.isnull().sum().to_dict())


In [ ]:
# ── CELL 7: Join coordinates onto well table ──────────────────────────────────

df_well_coords = df_well.merge(coords, on="API_NO", how="left")
del coords
gc.collect()

matched = df_well_coords["LATITUDE"].notna().sum()
total   = len(df_well_coords)
log.info("Coordinate match rate: %s / %s wells (%.1f%%)",
         f"{matched:,}", f"{total:,}", matched / total * 100)

print("\nSample with coordinates:")
print(df_well_coords[["DISTRICT_NO", "LEASE_NO", "WELL_NO",
                       "API_NO", "COUNTY_NAME",
                       "LATITUDE", "LONGITUDE"]].dropna().head(10))


In [ ]:
# ── CELL 8: Average coordinates per lease ────────────────────────────────────
# One representative lat/long per lease (average of all wells on that lease).

lease_coords = (
    df_well_coords
    .groupby(["DISTRICT_NO", "LEASE_NO"])[["LATITUDE", "LONGITUDE"]]
    .mean()
    .reset_index()
)
del df_well_coords, df_well
gc.collect()

log.info("✓ Lease coordinates: %s unique leases", f"{len(lease_coords):,}")
print("Nulls:", lease_coords.isnull().sum().to_dict())
print("\nSample:")
print(lease_coords.head(10))


In [ ]:
# ── CELL 9: Join lease coordinates onto production data ───────────────────────

missing = [c for c in ["DISTRICT_NO", "LEASE_NO"] if c not in df_cycle.columns]
if missing:
    log.error("Missing join columns in df_cycle: %s", missing)
    log.error("df_cycle columns: %s", df_cycle.columns.tolist())
else:
    df_cycle_geo = df_cycle.merge(lease_coords, on=["DISTRICT_NO", "LEASE_NO"], how="left")
    del lease_coords
    gc.collect()

    matched = df_cycle_geo["LATITUDE"].notna().sum()
    total   = len(df_cycle_geo)
    log.info("✓ Merged: %s rows x %s cols", f"{len(df_cycle_geo):,}", len(df_cycle_geo.columns))
    log.info("  Rows with coordinates: %s / %s (%.1f%%)",
             f"{matched:,}", f"{total:,}", matched / total * 100)

    print("\nSample:")
    print(df_cycle_geo[["DISTRICT_NO", "CYCLE_YEAR_MONTH",
                         "LEASE_OIL_PROD_VOL",
                         "LATITUDE", "LONGITUDE"]].dropna().head(10))


In [ ]:
# ── CELL 10: Save final output ────────────────────────────────────────────────

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

path = out / f"og_lease_cycle_with_coords.{FORMAT}"
if FORMAT == "parquet":
    df_cycle_geo.to_parquet(path, index=False)
else:
    df_cycle_geo.to_csv(path, index=False)

size_mb = path.stat().st_size / 1_048_576
log.info("✓ Saved %s  (%.1f MB)", path.name, size_mb)

print("\nFinal columns:", df_cycle_geo.columns.tolist())
print("\nAll files in output folder:")
for f in sorted(out.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1_048_576:.1f} MB)")


In [ ]:
# ── CELL 11: Quick sanity check plot ─────────────────────────────────────────
# Scatter plot of well locations — should look like Texas.

import matplotlib.pyplot as plt

sample = df_cycle_geo[["LATITUDE", "LONGITUDE"]].dropna().drop_duplicates()

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(sample["LONGITUDE"], sample["LATITUDE"],
           s=0.5, alpha=0.3, color="steelblue")
ax.set_title("Texas Oil Lease Locations — RRC PDQ + Well Layers")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(-107, -93)
ax.set_ylim(25, 37)
plt.tight_layout()
plt.show()

print(f"Unique coordinate pairs plotted: {len(sample):,}")
